# Mass defect plot (animated)

Visualizes all peaks of each sample as a frame in an animated scatter plot. The x-axis is m/z, the y-axis is mass defect (nominal mass − exact mass), and marker size represents log-intensity.

**NOTE:** Rendering may be slow if many samples are loaded.


### Load peaks


In [ ]:
from mascope_sdk import MascopeClient


mascope = MascopeClient()

# Load peaks (adjust workspace/batches to your data)
peaks = mascope.load_peaks(workspace="My Workspace", batches="My Batch")
peaks

### Compute mass defect


In [ ]:
import numpy as np


df = peaks.sort_values(["datetime_utc", "mz"]).copy()
df["mass_defect"] = df["mz"] - df["mz"].round()
df["log_intensity"] = np.log10(df["area"].clip(lower=1))  # or "height"

### Add `matched` column to use as a color category


In [ ]:
df["matched"] = df["target_isotope_formula"].notna()
df = df.fillna({"target_isotope_formula": "Unmatched"})

### Animated scatter plot

Drag the slider to move between samples, or press play to animate.


In [ ]:
import plotly.express as px
import plotly.io as pio


pio.templates.default = "plotly_dark"  # or "plotly_white"

fig = px.scatter(
    df,
    x="mz",
    y="mass_defect",
    color="matched",
    animation_frame="sample_item_name",
    animation_group="peak_id",
    size="log_intensity",
    size_max=15,
    hover_data=["target_ion_formula", "area"],
    range_x=[df["mz"].min() - 10, df["mz"].max() + 10],
    range_y=[df["mass_defect"].min() - 0.1, df["mass_defect"].max() + 0.1],
    width=900,
    height=700,
    labels={"mz": "m/z", "mass_defect": "Mass defect", "sample_item_name": "Sample"},
)
fig.update_layout(template="plotly_white")
fig.show()